##### Neural Network

The goal of this file is to implement a neural network model. 

In [8]:
# Columns for reference
columns = ['un_profitability', 'vote_average', 'vote_count', 'revenue', 'runtime', 'budget', 'popularity', 'actor_avg', 'actor_med', 'actor_dev', 'production_avg', 'production_med', 'production_dev', 'release_year', 'release_month', 'original_title_matches', 'profit', 'original_language_english', 'american_film', 'english_language']

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------------
# 1.  Load & basic prep (same as before)
# ------------------------------------------------------------------
df = pd.read_csv('movie-data/cleaned_analysis_data.csv')
df = df.drop(columns=['budget', 'profit', 'revenue', 'title', 'release_date'])

X_full = df.drop(columns=['un_profitability']).values
y_full = df['un_profitability'].values      # binary 0 / 1

# ------------------------------------------------------------------
# 2.  Helper to (re)build your network exactly once per split
# ------------------------------------------------------------------
def build_model(input_dim):
    model = Sequential([
        Dense(64, input_shape=(input_dim,), activation='relu'),
        Dropout(0.30),
        Dense(32, activation='relu'),
        Dropout(0.30),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# ------------------------------------------------------------------
# 3.  k-time hold-out loop
# ------------------------------------------------------------------
k = 10
sss = StratifiedShuffleSplit(n_splits=k, test_size=0.30, random_state=42)

acc, prec, rec, f1 = [], [], [], []

for tr_idx, te_idx in sss.split(X_full, y_full):
    X_train, X_test = X_full[tr_idx], X_full[te_idx]
    y_train, y_test = y_full[tr_idx], y_full[te_idx]

    # ---- standardize inside the split to avoid leakage ----
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ---- build, fit, predict ----
    nn = build_model(input_dim=X_train.shape[1])

    nn.fit(
        X_train, y_train,
        epochs=500,
        batch_size=32,
        validation_split=0.20,
        verbose=0,
        callbacks=[EarlyStopping(patience=10, restore_best_weights=True)]
    )

    y_prob = nn.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    # ---- macro averages give meaningful scores even if one class is rare ----
    acc.append( accuracy_score (y_test, y_pred) )
    prec.append( precision_score(y_test, y_pred, average='macro', zero_division=0) )
    rec.append(  recall_score   (y_test, y_pred, average='macro', zero_division=0) )
    f1.append(   f1_score      (y_test, y_pred, average='macro', zero_division=0) )

# ------------------------------------------------------------------
# 4.  Summary (µ ± σ) to copy into your LaTeX table
# ------------------------------------------------------------------
def show(name, arr):
    print(f"{name:<9}: {np.mean(arr):.3f} ± {np.std(arr):.3f}")

show("Accuracy",  acc)
show("Precision", prec)
show("Recall",    rec)
show("F1-score",  f1)


c:\Users\Chris\OneDrive\Desktop\Data Science\Data-Science-Project\env-desktop\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Chris\OneDrive\Desktop\Data Science\Data-Science-Project\env-desktop\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Chris\OneDrive\Desktop\Data Science\Data-Science-Project\env-desktop\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequentia

Accuracy : 0.761 ± 0.005
Precision: 0.752 ± 0.007
Recall   : 0.701 ± 0.007
F1-score : 0.712 ± 0.007
